In [11]:
import osmnx as ox
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
import ast
from shapely.geometry import LineString
import math
from haversine import haversine
from typing import List, Tuple, Optional
import math
import networkx as nx
import csv

# mappymatch imports
from mappymatch.constructs.trace import Trace
from mappymatch.maps.nx.nx_map import NxMap
from mappymatch.matchers.lcss.lcss import LCSSMatcher
pd.set_option('future.no_silent_downcasting', True)

In [2]:
# Define bbox manually
north = 41.35771290159195
south = 40.9990363866696
west  = -8.739966476287791
east  = -8.459738123284369


bbox = (west, south, east, north)
# Fetch network from the bounding box
G_bbox = ox.graph_from_bbox(
    bbox=bbox,
    network_type="drive",   # can be 'drive', 'walk', etc.
    simplify=True,
)

# Project to local CRS (UTM Zone 29N, or Mercator 3857)
porto_crs = 3857
G = ox.project_graph(G_bbox, to_crs=porto_crs)

# Add geometry to edges that don't have it
for u, v, k, data in G.edges(keys=True, data=True):
    if 'geometry' not in data:
        point_u = (G.nodes[u]['x'], G.nodes[u]['y']) # x is long, y is lat
        point_v = (G.nodes[v]['x'], G.nodes[v]['y'])
        data['geometry'] = LineString([point_u, point_v])
        

In [3]:
# Read trajectory data of the form long, lat
traj_df = pd.read_csv("train_1500.csv", index_col=0)

polylines = traj_df["POLYLINE"].to_numpy()
trajectories = []

for line in polylines:
    traj = ast.literal_eval(line)  # list of (lon, lat)
    trajectories.append(traj)


# Cleaning Trajectories

### Part 1: removing noise

We use a simple distance threshold to remove noise.

In [4]:
def distance_threshold_filter(traj, max_dist=250, traj_num=None):
    if len(traj) < 2:
        return traj

    def forward_pass(traj):
        filtered = [traj[0]]
        curr = 0
        next = 1
        num_points = len(traj)

        while next < num_points:
            # distance to current point
            dist = haversine((traj[curr][1], traj[curr][0]), (traj[next][1], traj[next][0]), unit="m")
            if dist <= max_dist:
                filtered.append(traj[next])
                curr = next  # move current pointer
            # else: skip next point (outlier)
            next += 1

        return filtered

    # forward pass
    fwd_filtered = forward_pass(traj)
    # backward pass
    bwd_filtered = forward_pass(traj[::-1])
    bwd_filtered = bwd_filtered[::-1]

    if len(fwd_filtered) == len(bwd_filtered):
        return fwd_filtered
    
    print(f"[Trajectory {traj_num}] Forward/backward pass differ: "
            f"forward={len(fwd_filtered)}, backward={len(bwd_filtered)}")

    return fwd_filtered if len(fwd_filtered) > len(bwd_filtered) else bwd_filtered 


clean_trajectories = []
num_filtered = 0
total_removed = 0
max_dist = 800  # meters, adjust as needed

for idx, traj in enumerate(trajectories):
    cleaned = distance_threshold_filter(traj, max_dist=max_dist, traj_num=idx)
    clean_trajectories.append(cleaned)

    if len(cleaned) < len(traj):
        num_filtered += 1
        total_removed += len(traj) - len(cleaned)

print(f"✅ {num_filtered}/{len(trajectories)} trajectories had points removed, "
      f"total of {total_removed} points filtered.")

[Trajectory 15] Forward/backward pass differ: forward=3, backward=59
[Trajectory 184] Forward/backward pass differ: forward=45, backward=34
[Trajectory 216] Forward/backward pass differ: forward=3, backward=78
[Trajectory 224] Forward/backward pass differ: forward=44, backward=2
[Trajectory 302] Forward/backward pass differ: forward=10, backward=72
[Trajectory 376] Forward/backward pass differ: forward=25, backward=35
[Trajectory 390] Forward/backward pass differ: forward=6, backward=76
[Trajectory 456] Forward/backward pass differ: forward=56, backward=28
[Trajectory 697] Forward/backward pass differ: forward=52, backward=4
[Trajectory 730] Forward/backward pass differ: forward=14, backward=33
[Trajectory 740] Forward/backward pass differ: forward=15, backward=10
[Trajectory 766] Forward/backward pass differ: forward=15, backward=16
[Trajectory 846] Forward/backward pass differ: forward=65, backward=56
[Trajectory 859] Forward/backward pass differ: forward=9, backward=54
[Trajectory 9

### Part 2: Detecting loops

LCSS does not give reliable results on loops, there each trajectory is broken down so that it has no loops

In [5]:

def cumulative_step_lengths(traj: List[Tuple[float,float]]) -> List[float]:
    """Compute cumulative path distances between GPS points."""
    n = len(traj)
    cum = [0.0] * n
    if n < 2:
        return cum
    for idx in range(1, n):
        d = haversine((traj[idx-1][1], traj[idx-1][0]),
                      (traj[idx][1],   traj[idx][0]),
                      unit="m")
        cum[idx] = cum[idx-1] + d
    return cum

def bearing(p1: Tuple[float, float], p2: Tuple[float, float]) -> float:
    """Calculate bearing between two GPS points in degrees."""
    lat1, lon1 = math.radians(p1[1]), math.radians(p1[0])
    lat2, lon2 = math.radians(p2[1]), math.radians(p2[0])
    
    dlon = lon2 - lon1
    x = math.sin(dlon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(dlon)
    initial_bearing = math.atan2(x, y)
    return (math.degrees(initial_bearing) + 360) % 360

def segments_intersect(p1: Tuple[float, float], p2: Tuple[float, float], 
                      p3: Tuple[float, float], p4: Tuple[float, float], 
                      tol: float = 1e-9) -> bool:
    """Check if two segments (p1-p2 and p3-p4) intersect."""
    def cross(o: Tuple[float, float], a: Tuple[float, float], b: Tuple[float, float]) -> float:
        return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])
    
    # Bounding box quick rejection
    if (max(p1[0], p2[0]) < min(p3[0], p4[0]) or 
        max(p3[0], p4[0]) < min(p1[0], p2[0]) or
        max(p1[1], p2[1]) < min(p3[1], p4[1]) or
        max(p3[1], p4[1]) < min(p1[1], p2[1])):
        return False
    
    # Orientation tests
    o1 = cross(p1, p2, p3)
    o2 = cross(p1, p2, p4)
    o3 = cross(p3, p4, p1)
    o4 = cross(p3, p4, p2)
    
    # Proper intersection
    if o1 * o2 < -tol and o3 * o4 < -tol:
        return True
    
    # Collinear points (handle near-zero cases)
    if abs(o1) < tol and abs(o2) < tol and abs(o3) < tol and abs(o4) < tol:
        # Check if segments overlap
        def on_segment(p, q, r):
            return (min(p[0], r[0]) <= q[0] <= max(p[0], r[0]) and 
                    min(p[1], r[1]) <= q[1] <= max(p[1], r[1]))
        
        return (on_segment(p1, p3, p2) or on_segment(p1, p4, p2) or
                on_segment(p3, p1, p4) or on_segment(p3, p2, p4))
    
    return False

def detect_innermost_loop_comprehensive(
    traj: List[Tuple[float, float]],
    r_loop: float = 25.0,
    min_gap: int = 3,
    min_len: int = 4,
    path_ratio: float = 2.0,
    bearing_threshold: float = 90.0,
    use_intersections: bool = True,
    use_proximity: bool = True
) -> Optional[Tuple[int, int]]:
    """
    Comprehensive loop detection using multiple strategies.
    Returns the innermost loop (smallest span) found.
    """
    n = len(traj)
    if n < min_len:
        return None
    
    cum = cumulative_step_lengths(traj)
    candidate_loops = []
    
    # Strategy 1: Proximity-based loop detection (Haversine)
    if use_proximity:
        for i in range(0, n - min_len):
            for j in range(i + max(min_gap, min_len - 1), n):
                disp = haversine((traj[i][1], traj[i][0]), 
                                (traj[j][1], traj[j][0]), unit="m")
                if disp > r_loop:
                    continue
                
                path_len = cum[j] - cum[i]
                if path_len / (disp + 1e-9) >= path_ratio:
                    candidate_loops.append((i, j, j - i, "proximity"))
                    break
    
    # Strategy 2: Segment intersection detection
    if use_intersections:
        for i in range(n - 1):
            for j in range(i + 2, n - 1):
                if segments_intersect(traj[i], traj[i+1], traj[j], traj[j+1]):
                    span = j - (i + 1)
                    if span >= min_len - 1:
                        candidate_loops.append((i + 1, j, span, "intersection"))
    
    # Strategy 3: Bearing change analysis
    for i in range(1, n - 1):
        bearing1 = bearing(traj[i-1], traj[i])
        bearing2 = bearing(traj[i], traj[i+1])
        bearing_change = abs((bearing2 - bearing1 + 180) % 360 - 180)
        
        if bearing_change > bearing_threshold:
            for j in range(i + min_gap, min(i + 20, n)):
                if j >= n:
                    break
                    
                return_bearing = bearing(traj[j-1], traj[j])
                if abs((return_bearing - bearing1 + 180) % 360 - 180) < 45:
                    disp = haversine((traj[i][1], traj[i][0]), 
                                    (traj[j][1], traj[j][0]), unit="m")
                    if disp < r_loop * 2:
                        candidate_loops.append((i, j, j - i, "bearing"))
                        break
    
    # Strategy 4: Exact/near-exact point duplicates
    duplicate_tol = 0.000001  # ~0.11 m tolerance
    for i in range(n):
        for j in range(i + min_gap, n):
            if (abs(traj[i][0] - traj[j][0]) < duplicate_tol and 
                abs(traj[i][1] - traj[j][1]) < duplicate_tol):
                candidate_loops.append((i, j, j - i, "duplicate"))
                break
    
    if not candidate_loops:
        return None
    
    candidate_loops.sort(key=lambda x: x[2])
    
    for i, (start, end, span, method) in enumerate(candidate_loops):
        is_innermost = True
        for other_start, other_end, other_span, _ in candidate_loops[:i]:
            if start <= other_start and end >= other_end and span > other_span:
                is_innermost = False
                break
        if is_innermost:
            return (start, end)
    
    return candidate_loops[0][:2]

def break_trajectory_recursive(
    traj: List[Tuple[float, float]],
    max_depth: int = 10,
    **loop_detection_kwargs
) -> List[List[Tuple[float, float]]]:
    """
    Recursively break trajectory at innermost loops until no loops remain or max_depth reached.
    Returns list of loop-free sub-trajectories.
    """
    if max_depth <= 0:
        return [traj]
    
    loop = detect_innermost_loop_comprehensive(traj, **loop_detection_kwargs)
    
    if loop is None:
        return [traj]  # No loops found
    
    start, end = loop
    mid = (start + end) // 2
    
    print(f"Breaking at loop {start}-{end} (mid: {mid}), trajectory length: {len(traj)}")
    
    # Recursively process both halves
    first_half = traj[:mid + 1]
    second_half = traj[mid:]
    
    result = []
    result.extend(break_trajectory_recursive(first_half, max_depth - 1, **loop_detection_kwargs))
    result.extend(break_trajectory_recursive(second_half, max_depth - 1, **loop_detection_kwargs))
    
    return result

# Main function 
def preprocess_trajectory_for_lcss(
    trajectory: List[Tuple[float, float]],
    r_loop: float = 25.0,
    min_gap: int = 3,
    min_len: int = 4
) -> List[List[Tuple[float, float]]]:
    """
    Preprocess trajectory by breaking it into loop-free segments for LCSS.
    """
    return break_trajectory_recursive(
        trajectory,
        r_loop=r_loop,
        min_gap=min_gap,
        min_len=min_len,
        path_ratio=2.0,
        bearing_threshold=90.0,
        use_intersections=True,
        use_proximity=True,
        max_depth=20
    )

In [6]:
# removing loops, converting each traj into a bunch of sub trajs

non_loop_trajectories_list = []

for traj in clean_trajectories:
    non_loop_trajectories_list.append(
        preprocess_trajectory_for_lcss(traj)
    )

Breaking at loop 30-33 (mid: 31), trajectory length: 63
Breaking at loop 0-3 (mid: 1), trajectory length: 39
Breaking at loop 0-3 (mid: 1), trajectory length: 38
Breaking at loop 0-3 (mid: 1), trajectory length: 37
Breaking at loop 29-32 (mid: 30), trajectory length: 34
Breaking at loop 25-28 (mid: 26), trajectory length: 59
Breaking at loop 0-3 (mid: 1), trajectory length: 33
Breaking at loop 10-13 (mid: 11), trajectory length: 32
Breaking at loop 1-4 (mid: 2), trajectory length: 12
Breaking at loop 5-8 (mid: 6), trajectory length: 10
Breaking at loop 46-49 (mid: 47), trajectory length: 65
Breaking at loop 38-41 (mid: 39), trajectory length: 48
Breaking at loop 4-8 (mid: 6), trajectory length: 9
Breaking at loop 0-6 (mid: 3), trajectory length: 7
Breaking at loop 2-5 (mid: 3), trajectory length: 18
Breaking at loop 1-4 (mid: 2), trajectory length: 15
Breaking at loop 1-4 (mid: 2), trajectory length: 13
Breaking at loop 1-8 (mid: 4), trajectory length: 11
Breaking at loop 13-16 (mid: 1

# Map Matching Using LCSS


In [7]:
matcher = LCSSMatcher(
    NxMap(G),
    distance_epsilon=100,       # default = 50
    similarity_cutoff=0.9,      # default = 0.9
    cutting_threshold=10,       # default = 10
    random_cuts=0,              # default = 0
    distance_threshold=500      # default = 10000
)

routes = []
matches_dfs = []

for i, traj_list in enumerate(non_loop_trajectories_list):
    trips = [
        gpd.GeoDataFrame(
            geometry=[Point(lon, lat) for lon, lat in traj],
            crs="EPSG:4326"
        ).to_crs(porto_crs)
        for traj in traj_list
    ]

    print(f"---- Processing trips {i+1} ----")

    this_routes = []
    this_matches_df = []

    for trip in trips:
        # Create Trace directly from GeoDataFrame (already in EPSG:3857)
        trace = Trace.from_geo_dataframe(trip, xy=False)

        # Perform matching
        try:
            match_result = matcher.match_trace(trace)
        except nx.NetworkXNoPath as e:
            print(f"⚠️  Skipping trip due to disconnected map segment: {e}")
            continue
        except Exception as e:
            print(f"⚠️  Other matching error: {e}")
            continue


        # Store per-coordinate matches
        this_matches_df.append(match_result.matches_to_dataframe())

        # Store matched path
        this_routes.append([route for route in match_result.path])
    # Safely combine sub-trips for this trajectory
    if this_routes:
        combined_route = [r for route_list in this_routes for r in route_list]
    else:
        combined_route = []

    if this_matches_df:
        combined_df = pd.concat(this_matches_df, ignore_index=True)
    else:
        combined_df = pd.DataFrame()

    # Always append, even if empty, to preserve index alignment
    routes.append(combined_route)
    matches_dfs.append(combined_df)

    print(f"Processed trajectory {i+1} with {len(combined_route)} matched segments.")


---- Processing trips 1 ----
Processed trajectory 1 with 21 matched segments.
---- Processing trips 2 ----
Processed trajectory 2 with 29 matched segments.
---- Processing trips 3 ----
Processed trajectory 3 with 62 matched segments.
---- Processing trips 4 ----
Processed trajectory 4 with 38 matched segments.
---- Processing trips 5 ----
Processed trajectory 5 with 44 matched segments.
---- Processing trips 6 ----
Processed trajectory 6 with 39 matched segments.
---- Processing trips 7 ----
Processed trajectory 7 with 61 matched segments.
---- Processing trips 8 ----
Processed trajectory 8 with 29 matched segments.
---- Processing trips 9 ----
Processed trajectory 9 with 55 matched segments.
---- Processing trips 10 ----
Processed trajectory 10 with 45 matched segments.
---- Processing trips 11 ----
Processed trajectory 11 with 19 matched segments.
---- Processing trips 12 ----
Processed trajectory 12 with 64 matched segments.
---- Processing trips 13 ----
Processed trajectory 13 with

# Storing results

In [10]:
"""
Checking for non uniqueness of osmid
"""

from collections import defaultdict

# Build a mapping from osmid to the edges that have it
osmid_to_edges = defaultdict(list)

for u, v, key, data in G.edges(keys=True, data=True):
    osmid = data.get("osmid")
    if osmid is None:
        continue
    # Handle multi-osmid edges
    if isinstance(osmid, list):
        for o in osmid:
            osmid_to_edges[o].append((u, v, key))
    else:
        osmid_to_edges[osmid].append((u, v, key))

# Total unique osmids
total_unique_osmids = len(osmid_to_edges)

# Non-unique osmids (appear on more than one edge)
non_unique_osmids = {o: edges for o, edges in osmid_to_edges.items() if len(edges) > 1}
num_non_unique = len(non_unique_osmids)

print(f"Total unique osmids: {total_unique_osmids}")
print(f"Non-unique osmids: {num_non_unique}")
print(f"Percentage non-unique: {num_non_unique / total_unique_osmids * 100:.2f}%")

# Optional: print a few examples
for osmid, edges in list(non_unique_osmids.items())[:10]:
    print(f"osmid {osmid} appears on {len(edges)} edges: {edges}")


Total unique osmids: 44841
Non-unique osmids: 30178
Percentage non-unique: 67.30%
osmid 1435552021 appears on 2 edges: [(12144320063, 10811800159, 0), (10811800159, 10811800162, 0)]
osmid 390107590 appears on 3 edges: [(25529420, 10012370912, 0), (10012370912, 25529422, 0), (25529422, 1329815314, 0)]
osmid 8366893 appears on 2 edges: [(25620516, 25620518, 0), (747467619, 25620516, 0)]
osmid 124539828 appears on 2 edges: [(1386051842, 1386051854, 0), (1386051854, 1386051842, 0)]
osmid 190128490 appears on 2 edges: [(1386051842, 1684844453, 0), (1684844453, 1386051842, 0)]
osmid 128825640 appears on 2 edges: [(2852022741, 3896080933, 0), (3896080933, 2852022741, 0)]
osmid 386286206 appears on 2 edges: [(2852022741, 3896080933, 0), (3896080933, 2852022741, 0)]
osmid 34607518 appears on 3 edges: [(25620570, 1106006327, 0), (1106006327, 25620759, 0), (25620653, 25620570, 0)]
osmid 229194693 appears on 2 edges: [(1106006327, 747467619, 0), (747467619, 25620516, 0)]
osmid 60175223 appears on 

### Storing routes

In [15]:
file_name = "part_3_routes_path.csv"

with open(file_name, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["route_tuples"])  # optional header
    
    for route in routes:
        if route:  # Non-empty route
            # Convert each road's (start, end, key) tuple to a string
            row = [str((road.road_id.start, road.road_id.end, road.road_id.key)) for road in route]
            writer.writerow(row)
        else:
            # Empty route -> write an empty line
            writer.writerow([])


### Storing road links matched to gps points

In [19]:
file_name_matches = "part_3_matches.csv"

with open(file_name_matches, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["matched_road_ids"])  # optional header
    
    for df in matches_dfs:
        if df is not None and not df.empty:
            row = []
            for rid in df["road_id"]:
                if rid is not None and not isinstance(rid, float):  # skip NaN/None
                    row.append(str((rid.start, rid.end, rid.key)))
            writer.writerow(row)
        else:
            writer.writerow([])
